# 🔢 Dạy máy tính nghe số

Notebook này dạy máy tính **nghe một đoạn ghi âm và đoán người ta nói số mấy (0–9)**.

**Cách dùng:** bấm **Run All** ở thanh công cụ phía trên, rồi kéo xuống xem kết quả. Toàn bộ mất khoảng 1–2 phút.

| Bước | Việc làm |
|---|---|
| 1 | Tìm dữ liệu |
| 2 | Xem qua dữ liệu và nghe thử |
| 3 | Biến âm thanh thành con số |
| 4 | Dạy máy và cho máy "đi thi" |
| 5 | Thử thách khó hơn: giọng người lạ |
| 6 | Tự thử với một file bất kỳ |

## Bước 1 — Tìm dữ liệu
Đoạn code này tự tìm các file ghi âm. Nếu báo lỗi, làm theo hướng dẫn trong lỗi.

In [ ]:
import re
from pathlib import Path

import numpy as np
from scipy.io import wavfile

# Tên file dạng: {số}_{người đọc}_{lần đọc}.wav  ví dụ 7_jackson_32.wav
PATTERN = re.compile(r"^(\d)_([a-z]+)_(\d+)\.wav$")

files = {}
for root in [Path("/kaggle/input"), Path("data")]:
    if root.exists():
        for p in root.rglob("*.wav"):
            if PATTERN.match(p.name):
                files.setdefault(p.name, p)   # bỏ file trùng tên

if not files:
    raise FileNotFoundError(
        "❌ Chưa thấy dữ liệu.\n"
        "Cách sửa: ở cột bên phải bấm 'Add Input', tìm 'free spoken digit dataset fsdd', "
        "bấm dấu + để thêm, rồi bấm Run All lại."
    )

recordings = []
for name, path in sorted(files.items()):
    digit, speaker, index = PATTERN.match(name).groups()
    recordings.append({"path": path, "digit": int(digit), "speaker": speaker, "index": int(index)})

print(f"✅ Tìm thấy {len(recordings)} file ghi âm.")

## Bước 2 — Xem qua dữ liệu và nghe thử

In [ ]:
from collections import Counter

speakers = sorted({r["speaker"] for r in recordings})
print("Người đọc:", ", ".join(speakers))
print("Số file mỗi người:", dict(Counter(r["speaker"] for r in recordings)))
print("Số file mỗi chữ số:", dict(sorted(Counter(r["digit"] for r in recordings).items())))

In [ ]:
def load_audio(path):
    """Đọc file .wav thành dãy số trong khoảng -1 đến 1."""
    sr, x = wavfile.read(path)
    if x.ndim > 1:
        x = x.mean(axis=1)
    if np.issubdtype(x.dtype, np.integer):
        x = x / np.iinfo(x.dtype).max
    return x.astype(np.float32), sr

import matplotlib.pyplot as plt
from IPython.display import Audio, display

sample = recordings[0]
x, sr = load_audio(sample["path"])
print(f"File: {sample['path'].name}  →  số {sample['digit']}, người đọc {sample['speaker']}, dài {len(x)/sr:.2f} giây")
display(Audio(x, rate=sr))   # bấm ▶ để nghe

plt.figure(figsize=(8, 2))
plt.plot(np.arange(len(x)) / sr, x)
plt.title("Hình dạng sóng âm (trục ngang: giây)")
plt.xlabel("giây")
plt.show()

## Bước 3 — Biến âm thanh thành con số
Máy tính không "nghe" được như người. Ta đổi mỗi đoạn âm thanh thành **80 con số** mô tả âm sắc của giọng (gọi là **MFCC**), giống như "dấu vân tay" của âm thanh.

In [ ]:
try:
    import librosa
except ImportError:
    raise ImportError("❌ Thiếu thư viện librosa. Hãy thêm một ô code chứa dòng: !pip install librosa, chạy ô đó rồi Run All lại.")

def fingerprint(x, sr):
    x, _ = librosa.effects.trim(x, top_db=30)          # cắt khoảng lặng đầu/cuối
    m = librosa.feature.mfcc(y=x, sr=sr, n_mfcc=20, n_fft=256, hop_length=80)
    if m.shape[1] >= 3:
        d = librosa.feature.delta(m, width=min(9, m.shape[1] // 2 * 2 - 1))
    else:
        d = np.zeros_like(m)
    return np.concatenate([m.mean(1), m.std(1), d.mean(1), d.std(1)])

X = np.stack([fingerprint(*load_audio(r["path"])) for r in recordings])
y = np.array([r["digit"] for r in recordings])
print(f"✅ Đã biến {len(X)} file thành dấu vân tay, mỗi file {X.shape[1]} con số.")

## Bước 4 — Dạy máy và cho máy "đi thi"
- **Bài học (train):** máy được xem ví dụ *kèm đáp án*.
- **Bài thi (test):** giấu đáp án, xem máy đoán đúng bao nhiêu.

Theo quy ước chuẩn của bộ dữ liệu, **lần đọc 0–4** dùng làm bài thi, phần còn lại để học.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import ConfusionMatrixDisplay

def train_and_test(is_test, title):
    is_test = np.array(is_test)
    model = make_pipeline(StandardScaler(), SVC(C=10, gamma="scale"))
    model.fit(X[~is_test], y[~is_test])                 # học
    pred = model.predict(X[is_test])                    # đi thi
    acc = (pred == y[is_test]).mean()
    print(f"📚 Học {(~is_test).sum()} file  |  📝 Thi {is_test.sum()} file")
    print(f"🎯 Máy đoán đúng {acc:.1%}")
    ConfusionMatrixDisplay.from_predictions(y[is_test], pred, cmap="Blues")
    plt.title(title)
    plt.xlabel("Máy đoán là số")
    plt.ylabel("Số đúng")
    plt.show()
    return model

model = train_and_test([r["index"] < 5 for r in recordings], "Bài thi chuẩn")

**Cách đọc bảng màu:** mỗi hàng là số đúng, mỗi cột là số máy đoán.
Ô trên **đường chéo** là đoán đúng. Ô đậm màu nằm *ngoài* đường chéo cho biết máy hay nhầm hai số nào với nhau.

## Bước 5 — Thử thách khó hơn: giọng người lạ
Lần này ta giấu **toàn bộ giọng của một người**, để xem máy có nhận ra giọng nó *chưa từng nghe* không. Kết quả thường thấp hơn, và đó là điều bình thường.

👉 Muốn thử người khác thì đổi tên trong dòng `held_out = ...` (chọn một tên trong danh sách ở Bước 2).

In [ ]:
held_out = speakers[-1]
print("Người bị giấu:", held_out)
train_and_test([r["speaker"] == held_out for r in recordings], f"Giọng người lạ: {held_out}")

## Bước 6 — Tự thử với một file bất kỳ
Đổi số trong `recordings[...]` để nghe một file khác và xem máy đoán gì.

In [ ]:
r = recordings[123]
x, sr = load_audio(r["path"])
display(Audio(x, rate=sr))
guess = model.predict([fingerprint(x, sr)])[0]
print(f"Đáp án: {r['digit']}  |  Máy đoán: {guess}  {'✅' if guess == r['digit'] else '❌'}")

## 🎉 Xong!
Bạn vừa dạy máy tính nhận ra giọng nói. Một vài ý để thử tiếp:
- Đổi `C=10` ở Bước 4 thành `C=1` hoặc `C=100` rồi xem độ chính xác thay đổi thế nào.
- Ở Bước 5, thử giấu từng người một, xem giọng ai khó nhận ra nhất.